# Late Interaction Retrieval - 04: What it costs

> **MLCourse - Agentic AI - Advanced RAG - Module 15**

Notebook 03 measured quality. This one measures the bill, because the bill is
the reason late interaction is not simply the default.

### What you will learn

1. The storage arithmetic: exactly how much bigger a token index is.
2. Query latency: how MaxSim scales with corpus size, measured.
3. The two compression tricks real ColBERT uses - **dimension reduction** and
   **quantization** - and what each costs in quality.
4. A decision rule for when this is worth paying for.

Still no LLM calls. Everything below is measured on this machine, so the
absolute numbers are specific to it; the *ratios* are the transferable part.

### Setup


In [ ]:
import time
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

MODEL = SentenceTransformer("all-MiniLM-L6-v2")
DIM = MODEL.get_embedding_dimension()


def normalize(mat):
    arr = mat.detach().cpu().numpy() if isinstance(mat, torch.Tensor) else np.asarray(mat)
    arr = arr.astype(np.float32)
    return arr / (np.linalg.norm(arr, axis=-1, keepdims=True) + 1e-12)


# A handful of realistic-length passages to measure token counts against.
PASSAGES = [
    "Photosynthesis converts light energy into chemical energy stored as glucose. "
    "It takes place in the chloroplasts of plant cells, where the pigment chlorophyll "
    "absorbs light in the blue and red parts of the spectrum.",
    "TCP guarantees ordered and reliable delivery of a byte stream between two hosts. "
    "It achieves this with sequence numbers, acknowledgements, and retransmission of "
    "packets that are lost or corrupted in transit.",
    "A Python generator function uses the yield keyword to produce values one at a time. "
    "Because it does not build the whole sequence in memory, it can represent very large "
    "or even infinite sequences cheaply.",
    "The Domain Name System translates human-readable domain names into IP addresses. "
    "Resolvers query a hierarchy of servers, starting at the root, and cache the answers "
    "for a period controlled by each record's time-to-live.",
]

tok_counts = [len(MODEL.tokenizer(p)["input_ids"]) for p in PASSAGES]
print(f"{'passage':<10} {'chars':>7} {'tokens':>8}")
print("-" * 28)
for i, (p, n) in enumerate(zip(PASSAGES, tok_counts)):
    print(f"P{i:<9} {len(p):>7} {n:>8}")
print("-" * 28)
AVG_TOKENS = float(np.mean(tok_counts))
print(f"{'average':<10} {'':>7} {AVG_TOKENS:>8.1f}")


### 1. The storage arithmetic

This is the whole objection to late interaction, and it is worth writing out
rather than hand-waving at.

A bi-encoder stores **one vector per passage**:

```
bytes_per_passage = dim x bytes_per_float
```

A late-interaction index stores **one vector per token**:

```
bytes_per_passage = tokens x dim x bytes_per_float
```

The ratio is just `tokens`. There is no cleverness in the formula - you are
storing a matrix where you stored a vector, and the multiplier is however many
tokens your chunks contain. For typical RAG chunk sizes that is **50-200×**.

### Storage, measured on real encodings


In [ ]:
pooled = MODEL.encode(PASSAGES, normalize_embeddings=True)
token_mats = [normalize(m) for m in MODEL.encode(PASSAGES, output_value="token_embeddings")]

pooled_bytes = pooled.nbytes
token_bytes = sum(m.nbytes for m in token_mats)

print(f"{len(PASSAGES)} passages, float32, {DIM} dims\n")
print(f"single-vector index : {pooled_bytes:>10,} bytes"
      f"  ({pooled_bytes / len(PASSAGES):>8,.0f} per passage)")
print(f"token-level index   : {token_bytes:>10,} bytes"
      f"  ({token_bytes / len(PASSAGES):>8,.0f} per passage)")
print(f"\nratio: {token_bytes / pooled_bytes:.1f}x  (= average tokens per passage)")

# Now extrapolate to corpus sizes people actually run.
print(f"\n{'corpus size':>14} {'single-vector':>16} {'late interaction':>18}")
print("-" * 52)
for n in (10_000, 1_000_000, 100_000_000):
    a = n * DIM * 4
    b = n * AVG_TOKENS * DIM * 4
    print(f"{n:>14,} {a / 1e9:>13.2f} GB {b / 1e9:>15.2f} GB")


### Reading those numbers

A million passages is an ordinary, unremarkable corpus - one mid-sized
documentation site, or a few years of support tickets.

At a million passages the single-vector index fits comfortably in RAM on a
laptop. The uncompressed late-interaction index does not fit in RAM on most
servers. That is the entire practical problem, stated in one comparison.

Real ColBERT deployments do not store what we just measured. They attack it
from two directions at once.

### 2. Compression trick 1 - dimension reduction

ColBERT does not use the model's native hidden size for its token vectors. It
adds a linear projection down to a much smaller dimension - **128** in
ColBERTv1/v2, and as low as **96 or 64** in later work - and trains the model
with that projection in place, so the model learns to pack what matters into
the smaller space.

We cannot train, so we will approximate the *storage and quality effect* with
PCA fitted on our own token vectors. This is a fair illustration of the
trade-off shape, and an unfair one for the absolute quality: a trained
projection loses far less than a post-hoc PCA does.

### Dimension reduction via PCA, with the quality cost measured


In [ ]:
from sklearn.decomposition import PCA

# Fit PCA on a decent pool of token vectors drawn from a bigger corpus.
CORPUS = PASSAGES + [
    "Cellular respiration breaks down glucose to release energy in the form of ATP.",
    "Mitochondria are the organelles in which cellular respiration takes place.",
    "UDP sends datagrams without any guarantee of delivery, ordering, or duplicate protection.",
    "HTTPS encrypts HTTP traffic using TLS so that intermediaries cannot read it.",
    "A list comprehension in Python builds a new list from an iterable in a single expression.",
    "Python dictionaries have preserved insertion order since version 3.7.",
    "Chlorophyll is the green pigment responsible for absorbing light during photosynthesis.",
    "Plants exchange gases with the atmosphere through small pores called stomata.",
]
QUESTIONS = [
    ("what stores energy from sunlight in plants", 0),
    ("which protocol retransmits lost packets", 1),
    ("how does Python represent infinite sequences", 2),
    ("what turns domain names into IP addresses", 3),
    ("where does cellular respiration happen", 5),
    ("what pigment absorbs light", 10),
]

mats = [normalize(m) for m in MODEL.encode(CORPUS, output_value="token_embeddings")]
all_tokens = np.vstack(mats)
print(f"fitting PCA on {all_tokens.shape[0]:,} token vectors of {DIM} dims\n")


def build_index(matrices, projector=None):
    """Pad a list of token matrices into (N, max_tok, d) plus a mask."""
    proj = [normalize(projector.transform(m)) if projector is not None else m for m in matrices]
    d = proj[0].shape[1]
    max_tok = max(m.shape[0] for m in proj)
    padded = np.zeros((len(proj), max_tok, d), dtype=np.float32)
    mask = np.zeros((len(proj), max_tok), dtype=bool)
    for i, m in enumerate(proj):
        padded[i, : m.shape[0]] = m
        mask[i, : m.shape[0]] = True
    return padded, mask


def mrr(padded, mask, projector=None):
    """Mean reciprocal rank of the gold passage over the question set."""
    rr = []
    for question, gold in QUESTIONS:
        q = normalize(MODEL.encode(question, output_value="token_embeddings"))
        if projector is not None:
            q = normalize(projector.transform(q))
        sim = np.einsum("ik,djk->dij", q, padded)
        sim = np.where(mask[:, None, :], sim, -np.inf)
        scores = sim.max(axis=2).sum(axis=1)
        rank = int(np.where(np.argsort(-scores) == gold)[0][0]) + 1
        rr.append(1.0 / rank)
    return float(np.mean(rr))


base_padded, base_mask = build_index(mats)
base_mrr = mrr(base_padded, base_mask)
base_bytes = int(base_mask.sum()) * DIM * 4

print(f"{'dims':>6} {'bytes/passage':>15} {'vs 384':>8} {'MRR':>7}")
print("-" * 40)
print(f"{DIM:>6} {base_bytes / len(CORPUS):>15,.0f} {'1.0x':>8} {base_mrr:>7.3f}")

for d in (128, 64, 32):
    pca = PCA(n_components=d, random_state=0).fit(all_tokens)
    padded, mask = build_index(mats, projector=pca)
    m = mrr(padded, mask, projector=pca)
    nbytes = int(mask.sum()) * d * 4
    print(f"{d:>6} {nbytes / len(CORPUS):>15,.0f} {DIM / d:>7.1f}x {m:>7.3f}")


### 3. Compression trick 2 - quantization

Dimension reduction shrinks the *number* of values. Quantization shrinks the
*size* of each one.

Our vectors are `float32` - 4 bytes per dimension. But these are L2-normalised
vectors, so every component lies in `[-1, 1]`, and we do not need anything
like 32 bits of precision to represent a coordinate in that range well enough
to rank documents.

ColBERTv2's PLAID index goes to roughly **2 bits per dimension** using
residual compression against learned centroids. We will demonstrate the
simpler idea - plain scalar quantization to 8 and 4 bits - and measure what
each costs.

### Scalar quantization, with the quality cost measured


In [ ]:
def quantize(arr, bits):
    """Scalar-quantize values in [-1, 1] to `bits` bits, then dequantize.

    Storage would be `bits` per dimension; we immediately decode back to
    float32 so we can score with the same code and see the quality effect
    in isolation from any indexing machinery.
    """
    levels = 2 ** bits - 1
    clipped = np.clip(arr, -1.0, 1.0)
    codes = np.round((clipped + 1.0) / 2.0 * levels)        # -> integers 0..levels
    return (codes / levels) * 2.0 - 1.0                     # -> back to [-1, 1]


print(f"{'precision':<14} {'bytes/passage':>15} {'vs fp32':>9} {'MRR':>7}")
print("-" * 49)
print(f"{'float32':<14} {base_bytes / len(CORPUS):>15,.0f} {'1.0x':>9} {base_mrr:>7.3f}")

for bits in (8, 4, 2):
    q_mats = [normalize(quantize(m, bits)) for m in mats]
    padded, mask = build_index(q_mats)
    m = mrr(padded, mask)
    nbytes = int(mask.sum()) * DIM * bits / 8
    print(f"{f'int{bits}':<14} {nbytes / len(CORPUS):>15,.0f} {32 / bits:>8.0f}x {m:>7.3f}")


### Both tricks together: the configuration ColBERT actually ships


In [ ]:
print(f"{'configuration':<28} {'bytes/passage':>15} {'vs baseline':>13} {'MRR':>7}")
print("-" * 66)

single_vec_bytes = DIM * 4
print(f"{'single vector (fp32)':<28} {single_vec_bytes:>15,.0f} "
      f"{single_vec_bytes / base_bytes * len(CORPUS):>12.3f}x {'n/a':>7}")
print(f"{'late interaction, fp32, 384d':<28} {base_bytes / len(CORPUS):>15,.0f} "
      f"{'1.000x':>13} {base_mrr:>7.3f}")

for d, bits in ((128, 8), (128, 2), (64, 2)):
    pca = PCA(n_components=d, random_state=0).fit(all_tokens)
    proj = [normalize(quantize(normalize(pca.transform(m)), bits)) for m in mats]
    padded, mask = build_index(proj)

    rr = []
    for question, gold in QUESTIONS:
        q = normalize(quantize(normalize(pca.transform(
            normalize(MODEL.encode(question, output_value="token_embeddings")))), bits))
        sim = np.einsum("ik,djk->dij", q, padded)
        sim = np.where(mask[:, None, :], sim, -np.inf)
        scores = sim.max(axis=2).sum(axis=1)
        rr.append(1.0 / (int(np.where(np.argsort(-scores) == gold)[0][0]) + 1))

    nbytes = int(mask.sum()) * d * bits / 8
    label = f"late interaction, {d}d, int{bits}"
    print(f"{label:<28} {nbytes / len(CORPUS):>15,.0f} "
          f"{nbytes / base_bytes:>12.3f}x {np.mean(rr):>7.3f}")

print("\nThe last row is roughly ColBERTv2/PLAID's operating point.")


### What the compression table shows

Two things, and the second is the important one.

**First: the compression works.** Combining a projection to 128 dimensions
with aggressive quantization cuts the index by more than an order of magnitude
while MRR on this question set barely moves. That is why ColBERT is deployable
at all.

**Second - and be honest about this - our question set is far too small and
too easy to detect the quality loss that compression really causes.** Six
questions over sixteen passages will report "no degradation" for almost any
compression you throw at it. Published ColBERTv2 results measure this over
MS MARCO with millions of passages, and they do find a real (if small) cost.

So read the table as *"here is the storage arithmetic, and here is the shape
of the trade-off"*, not as *"2-bit quantization is free"*. It is not free. It
is cheap enough to be worth it, which is a different claim, and one this
notebook cannot verify at this scale.

### 4. Query latency

Storage is the headline cost, but query-time work grows too. Per candidate
document you now do a small matrix multiply instead of a single dot product.

Let's measure how that scales.

### Latency vs corpus size, measured


In [ ]:
rng = np.random.default_rng(0)
q_tokens = normalize(MODEL.encode(QUESTIONS[0][0], output_value="token_embeddings"))
n_q_tok, avg_tok = q_tokens.shape[0], 40      # 40 tokens/passage: a typical chunk

print(f"query has {n_q_tok} tokens; simulating {avg_tok} tokens per passage\n")
print(f"{'corpus':>10} {'dense (ms)':>12} {'maxsim (ms)':>13} {'ratio':>8}")
print("-" * 46)

for n in (1_000, 10_000, 50_000):
    # Random unit vectors stand in for a real index -- timing depends on array
    # shapes and memory traffic, not on the values.
    dense_idx = normalize(rng.standard_normal((n, DIM)).astype(np.float32))
    tok_idx = normalize(rng.standard_normal((n, avg_tok, DIM)).astype(np.float32))
    q_pooled = normalize(rng.standard_normal((1, DIM)).astype(np.float32))[0]

    t0 = time.perf_counter()
    for _ in range(5):
        _ = dense_idx @ q_pooled
    dense_ms = (time.perf_counter() - t0) / 5 * 1000

    t0 = time.perf_counter()
    for _ in range(5):
        sim = np.einsum("ik,djk->dij", q_tokens, tok_idx)
        _ = sim.max(axis=2).sum(axis=1)
    maxsim_ms = (time.perf_counter() - t0) / 5 * 1000

    print(f"{n:>10,} {dense_ms:>12.2f} {maxsim_ms:>13.2f} {maxsim_ms / dense_ms:>7.0f}x")

print("\nBoth are brute-force scans -- no ANN index on either side.")


### Reading the latency numbers

The ratio is large, and the honest framing needs two corrections.

**Correction 1: this comparison is unfair to nobody, but it is unrealistic.**
Neither side here uses an approximate-nearest-neighbour index. In production
the dense side would use HNSW or IVF and be far faster still. The
late-interaction side would use PLAID, which does not score every document
either - it uses the token centroids to shortlist candidates and only runs
full MaxSim on a few thousand of them. So neither column reflects a real
deployment.

**Correction 2: the ratio is the point, not the milliseconds.** MaxSim does
roughly `query_tokens x doc_tokens` more arithmetic per document than a dot
product. That is a constant factor of a few hundred to a few thousand, and it
does not go away with better engineering - it can only be avoided by scoring
fewer documents.

Which is exactly what PLAID does, and exactly what a retrieve-then-rerank
pipeline does. The general principle:

> Expensive scoring functions do not get cheaper. They get applied to fewer
> candidates.

Late interaction sits in the middle of this spectrum: cheap enough to apply to
a large shortlist, too expensive to apply to everything.

### 5. So when should you actually use this?

A decision checklist, in the order you should work through it.

**1. Have you measured a first-stage recall ceiling?** If you have not, stop.
Instrument Recall@100 first (`10_rag_evaluation`). Most RAG systems that feel
bad are not recall-limited - they are chunking-limited or prompt-limited.

**2. Have you tried the cheap fixes?** In rough order of effort-to-benefit:

| Fix | Cost | Covered in |
|---|---|---|
| Better chunking | low | `13_contextual_retrieval` |
| Hybrid BM25 + dense (RRF) | low | `01_hybrid_search` |
| Query rewriting / expansion | low | `12_query_transformation` |
| Cross-encoder reranking | low | [`11_reranking`](../11_reranking/README.md) |
| **Late interaction** | **high** | **this module** |

Late interaction is deliberately last. It is the only one on the list that
changes your *index format*, and that is a large, hard-to-reverse commitment.

**3. Is recall still your bottleneck after all that?** Then late interaction
is a reasonable answer, and you should use a real implementation rather than
the teaching version in this module.

**4. Can you afford the storage and the operational complexity?** Multiply
your passage count by your average token count. If that index does not fit
your budget after compression, the answer is no regardless of quality.

### Use a real implementation

If you get to step 3, do not ship what we built here. Use `pylate`,
`colbert-ai`, or `RAGatouille`, which give you a model *trained* for MaxSim
plus a PLAID index that makes it fast.

> **A note on this course's environment:** installing `pylate` here downgrades
> `torch`, `transformers` and `sentence-transformers`, which breaks other
> modules in this track. That is why this module reimplements MaxSim by hand.
> In a project of your own, in a clean virtual environment, install the real
> library.

### Key takeaways

- Storage is the real cost: **tokens × more bytes than a single-vector
  index**, typically 50-200×. At a million passages that is the difference
  between fitting in laptop RAM and not fitting on a server.
- **Dimension reduction** (384 → 128 or 64) and **quantization** (32 → 2 bits)
  together cut the index by more than an order of magnitude. Real ColBERTv2
  combines both.
- Our benchmark is **too small to measure the quality cost of compression**.
  The table shows the storage arithmetic and the trade-off shape, not proof
  that compression is free.
- Query-time work grows by roughly `query_tokens × doc_tokens` per document.
  Expensive scoring functions do not get cheaper - they get applied to fewer
  candidates (PLAID, or a rerank pipeline).
- **Try chunking, hybrid search, query rewriting, and cross-encoder reranking
  first.** Late interaction is the only one that changes your index format.
- If you do adopt it, use a real trained implementation, not this teaching
  reimplementation.

That completes module 15. Back to the
[module README](README.md), or on to the rest of
[Advanced RAG](../README.md).